# Dual-Head Audio Vectorizer: Full INT8 Quantization-Aware Training (QAT)

This notebook builds, trains, and exports an NPU-optimized audio vectorizer for **LiteRT**.

### Pipeline Architectural Constraints ("The Right Way"):
1. **Decoupled DSP:** Accepts precomputed log-Mel filterbank spectrograms `(50 frames x 80 bins)` representing exactly 1.0 second of audio. Fast Fourier Transforms and Mel-scale projections remain off the NPU.
2. **Strict Static Tiling:** Static tensor dimensions `(1, 50, 80)` prevent dynamic memory allocations and avoid delegate rejection.
3. **Zero Dynamic Operator Fallback:** Uses only standard 1D convolutions, Batch Normalization, and bounded `ReLU6` activations.
4. **Dual Embeddings:**
   - **Speaker Embedding:** 192-dimensional vector for cosine similarity tracking.
   - **Prosody Embedding:** 64-dimensional vector capturing cadence, energy, and inflection.
5. **Full INT8 Quantization-Aware Training (QAT):** Simulates 8-bit precision during fine-tuning and exports an integer-only graph (`int8` inputs, weights, activations, and outputs).

In [ ]:
!pip install -q tensorflow tensorflow-model-optimization numpy

In [ ]:
import os
import numpy as np
import tensorflow as tf
import tensorflow_model_optimization as tfmot

# Ensure deterministic operations
tf.random.set_seed(42)
np.random.seed(42)

print(f"TensorFlow Version: {tf.__version__}")
print(f"TFMOT Version: {tfmot.__version__}")

## 1. Constants & Static Dimensions

Audio framing parameters for 1.0-second sliding windows:
- Sample Rate: 16,000 Hz
- Window / FFT Size: 25 ms (400 samples)
- Hop Size: 20 ms (320 samples) $\rightarrow$ exactly 50 time steps per second
- Mel Bins: 80 filterbanks

In [ ]:
TIME_STEPS = 50
MEL_BINS = 80
SPEAKER_EMBEDDING_DIM = 192
PROSODY_EMBEDDING_DIM = 64
TRAIN_BATCH_SIZE = 32

## 2. NPU-Compatible Model Architecture

We use standard Conv1D layers with bounded `ReLU6` activations. Bounded activations avoid quantization clipping during fixed-point integer scaling.

In [ ]:
def build_base_vectorizer(time_steps=TIME_STEPS, mel_bins=MEL_BINS):
    inputs = tf.keras.Input(shape=(time_steps, mel_bins), name="mel_spectrogram_input")

    # Stage 1: Feature Extraction
    x = tf.keras.layers.Conv1D(128, kernel_size=5, strides=1, padding="same")(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU(max_value=6.0)(x)

    # Stage 2: Temporal Stride (Downsample time axis 50 -> 25)
    x = tf.keras.layers.Conv1D(192, kernel_size=3, strides=2, padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU(max_value=6.0)(x)

    # Stage 3: Dilated Temporal Context
    x = tf.keras.layers.Conv1D(256, kernel_size=3, dilation_rate=2, padding="same")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU(max_value=6.0)(x)

    # Stage 4: Global Temporal Aggregation (Pools over remaining 25 frames)
    pooled = tf.keras.layers.GlobalAveragePooling1D(name="temporal_pooling")(x)

    # Shared Bottleneck
    shared = tf.keras.layers.Dense(256)(pooled)
    shared = tf.keras.layers.BatchNormalization()(shared)
    shared = tf.keras.layers.ReLU(max_value=6.0)(shared)

    # Output Head 1: 192-d Speaker Embedding
    speaker_out = tf.keras.layers.Dense(
        SPEAKER_EMBEDDING_DIM,
        activation=None,
        name="speaker_embedding"
    )(shared)

    # Output Head 2: 64-d Prosody/Emotion Embedding
    prosody_out = tf.keras.layers.Dense(
        PROSODY_EMBEDDING_DIM,
        activation=None,
        name="prosody_embedding"
    )(shared)

    return tf.keras.Model(
        inputs=inputs,
        outputs=[speaker_out, prosody_out],
        name="dual_head_audio_vectorizer"
    )

base_model = build_base_vectorizer()
base_model.summary()

## 3. Synthetic Dataset Generation

We generate mock log-Mel features normalized between -4.0 and +4.0 (standard range for audio spectrograms) along with target vectors.

In [ ]:
NUM_SAMPLES = 2048

# Precomputed Log-Mel spectrogram inputs: (N, 50, 80)
X_train = np.random.normal(loc=0.0, scale=1.5, size=(NUM_SAMPLES, TIME_STEPS, MEL_BINS)).astype(np.float32)

# Speaker Ground Truth: 192-d normalized speaker vectors
y_speaker = np.random.normal(loc=0.0, scale=1.0, size=(NUM_SAMPLES, SPEAKER_EMBEDDING_DIM)).astype(np.float32)
y_speaker /= np.linalg.norm(y_speaker, axis=-1, keepdims=True)

# Prosody Ground Truth: 64-d prosody regression vectors
y_prosody = np.random.uniform(low=-1.0, high=1.0, size=(NUM_SAMPLES, PROSODY_EMBEDDING_DIM)).astype(np.float32)

train_ds = tf.data.Dataset.from_tensor_slices((
    X_train,
    {"speaker_embedding": y_speaker, "prosody_embedding": y_prosody}
)).batch(TRAIN_BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

## 4. Phase 1: Baseline Float32 Pre-Training

In [ ]:
base_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss={
        "speaker_embedding": tf.keras.losses.CosineSimilarity(),
        "prosody_embedding": tf.keras.losses.MeanSquaredError()
    },
    loss_weights={
        "speaker_embedding": 1.0,
        "prosody_embedding": 0.5
    }
)

print("Training baseline Float32 model...")
base_model.fit(train_ds, epochs=3, verbose=1)

## 5. Phase 2: Quantization-Aware Training (QAT)

We wrap the pre-trained model with `tfmot.quantization.keras.quantize_model`. This injects fake-quantization nodes into the weights and activation layers so the network learns to compensate for 8-bit precision loss.

In [ ]:
quantize_model = tfmot.quantization.keras.quantize_model
qat_model = quantize_model(base_model)

# Compile QAT model with a smaller learning rate for fine-tuning
qat_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss={
        "speaker_embedding": tf.keras.losses.CosineSimilarity(),
        "prosody_embedding": tf.keras.losses.MeanSquaredError()
    },
    loss_weights={
        "speaker_embedding": 1.0,
        "prosody_embedding": 0.5
    }
)

print("Fine-tuning with Quantization-Aware Training...")
qat_model.fit(train_ds, epochs=2, verbose=1)

## 6. Phase 3: Export Full INT8 Model for NPU Deployment

To ensure compatibility with NPU compilers (such as Qualcomm QNN and MediaTek NeuroPilot):
1. Wrap the QAT model in an export container with an explicit static shape `(1, 50, 80)`.
2. Provide a calibrated representative dataset to compute per-tensor activation scales and zero points.
3. Set both `inference_input_type` and `inference_output_type` to `tf.int8`.

In [ ]:
# Step A: Freeze batch size to exactly 1 for mobile runtime
inference_inputs = tf.keras.Input(shape=(TIME_STEPS, MEL_BINS), batch_size=1, name="mel_spectrogram_input")
inference_outputs = qat_model(inference_inputs)
export_qat_model = tf.keras.Model(inputs=inference_inputs, outputs=inference_outputs)

# Step B: Representative dataset generator
def representative_dataset_gen():
    for i in range(128):
        sample = X_train[i:i+1] # Shape: (1, 50, 80)
        yield [sample]

# Step C: Configure LiteRT Converter for pure INT8 execution
converter = tf.lite.TFLiteConverter.from_keras_model(export_qat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen

# Strictly enforce INT8 operators across all subgraphs
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

# Enforce integer I/O to match NPU DirectByteBuffer bindings
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_int8_model = converter.convert()

# Save binary to disk
output_filename = "audio_vectorizer_qat_int8.tflite"
with open(output_filename, "wb") as f:
    f.write(tflite_int8_model)

size_kb = os.path.getsize(output_filename) / 1024.0
print(f"Export complete: '{output_filename}' ({size_kb:.2f} KB)")

## 7. Model Verification & Inspection

Verify that the model consists entirely of INT8 operations and check the scale and zero-point parameters needed for Android JNI quantization.

In [ ]:
interpreter = tf.lite.Interpreter(model_path=output_filename)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("=== INPUT TENSOR SPECIFICATION ===")
for inp in input_details:
    print(f"Name:       {inp['name']}")
    print(f"Shape:      {inp['shape']} (Static)")
    print(f"Dtype:      {inp['dtype']}")
    print(f"Quant Scale:      {inp['quantization_parameters']['scales']}")
    print(f"Quant Zero-Point: {inp['quantization_parameters']['zero_points']}\n")

print("=== OUTPUT TENSOR SPECIFICATION ===")
for out in output_details:
    print(f"Name:       {out['name']}")
    print(f"Shape:      {out['shape']} (Static)")
    print(f"Dtype:      {out['dtype']}")
    print(f"Quant Scale:      {out['quantization_parameters']['scales']}")
    print(f"Quant Zero-Point: {out['quantization_parameters']['zero_points']}\n")

# Ensure all tensors are strictly signed 8-bit integers
assert input_details[0]['dtype'] == np.int8, "Input must be INT8!"
for out in output_details:
    assert out['dtype'] == np.int8, "Outputs must be INT8!"

print("Model verification passed: 100% INT8 layout ready for LiteRT NPU acceleration.")

## 8. Android JNI Integration Helper

Use this helper script to dequantize the output vectors on the Android CPU once LiteRT completes an NPU burst.

In [ ]:
# Example JNI dequantization logic:
# real_value = (quantized_int8_value - zero_point) * scale

test_input = np.random.randint(-128, 127, size=(1, TIME_STEPS, MEL_BINS), dtype=np.int8)
interpreter.set_tensor(input_details[0]['index'], test_input)
interpreter.invoke()

raw_speaker_int8 = interpreter.get_tensor(output_details[0]['index'])
spk_scale = output_details[0]['quantization_parameters']['scales'][0]
spk_zp = output_details[0]['quantization_parameters']['zero_points'][0]

dequantized_speaker_vector = (raw_speaker_int8.astype(np.float32) - spk_zp) * spk_scale
print(f"Dequantized Speaker Vector Sample: {dequantized_speaker_vector[0][:5]}...")